# RAG systems

Author: Umberto Michelucci, umberto.michelucci@hslu.ch

## Minimal Retrieval-Augmented Generation (RAG)

This notebook shows the simplest possible implementation of a RAG system.

A RAG system works in 3 steps:

1. Store documents
2. Retrieve the most relevant documents for a question
3. Use an LLM to answer using the retrieved context

The goal is to help understand the core idea behind modern AI assistants that use external knowledge.

## Import libraries and install them

In [1]:
#!pip install sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
from transformers import pipeline

from pypdf import PdfReader

# Load model once
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load local model
generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)





/Users/umbertomichelucci/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 25527.01it/s]


In [3]:
def read_pdf(path):
    reader = PdfReader(path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

## Create a tiny knowledge base

This is our "database".

Each document contains one piece of bioengineering knowledge.

In a real RAG system:
- these could come from PDFs
- scientific papers
- textbooks
- websites
- databases

In [4]:
# 1. Tiny knowledge base
documents = [
    "Tissue engineering combines cells, scaffolds, and biochemical signals to repair or replace damaged tissue.",
    "Hydrogels are water-rich polymer networks often used as scaffolds because they can mimic soft biological tissues.",
    "CRISPR-Cas9 is a genome editing technology that can cut DNA at targeted locations.",
    "Biosensors combine a biological recognition element with a physical transducer to detect molecules.",
    "Drug delivery systems aim to release therapeutic molecules at the right place, time, and dose."
]

## Create embeddings

Embeddings convert text into numerical vectors.

Texts with similar meaning produce similar vectors.

The embedding model does NOT answer questions.
It only converts text into a mathematical representation.

In [5]:
def embed(text):
    embedding = model.encode(text)
    return np.array(embedding)

## Embed all documents

We now compute one embedding vector for each document.

This step is usually done once and stored in a vector database.

In [6]:
# 3. Embed all documents
doc_embeddings = [embed(doc) for doc in documents]

## Retrieve the most relevant documents

We now implement semantic search.

Steps:
1. Embed the user question
2. Compare the question vector to all document vectors
3. Compute cosine similarity
4. Return the most similar documents

Cosine similarity measures how close two vectors are.

In [7]:
# 4. Retrieve the most relevant documents
def retrieve(query, k=2):
    query_embedding = embed(query)

    similarities = []
    for doc_embedding in doc_embeddings:
        score = np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
        )
        similarities.append(score)

    top_indices = np.argsort(similarities)[-k:][::-1]

    return [documents[i] for i in top_indices]

In [8]:
def rag_answer(question):
    context_docs = retrieve(question, k=2)

    context = "\n".join(context_docs)

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=50,
        do_sample=False
    )

    generated_text = response[0]["generated_text"]

    # Remove prompt from output
    answer = generated_text[len(prompt):]

    return answer.strip()

In [9]:
# Example
question = "Why are hydrogels useful in tissue engineering?"

print(rag_answer(question))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Hydrogels are useful in tissue engineering because they can mimic the soft biological tissues. They can provide a scaffold for cells to grow on, and they can also provide a biochemical signal to stimulate cell


## Test retrieval

We first test ONLY the retrieval system.

This allows us to see:
- which documents were selected
- where the information comes from
- similarity scores

This is extremely important in RAG systems.

In [10]:
# 6. Try it
question = "Why are hydrogels useful in tissue engineering?"

answer = rag_answer(question)

print(answer)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hydrogels are useful in tissue engineering because they can mimic the soft biological tissues. They can provide a scaffold for cells to grow on, and they can also provide a biochemical signal to stimulate cell


## What happened internally?

The pipeline is:

Question
↓
Embedding
↓
Similarity search
↓
Retrieved documents
↓
LLM prompt with context
↓
Generated answer

This architecture is the foundation of many modern AI assistants.

# RAG from PDFs

In [11]:
pdf_1_text = read_pdf("paper1.pdf")
pdf_2_text = read_pdf("paper2.pdf")

In [12]:
documents = [
    pdf_1_text,
    pdf_2_text
]

In [13]:
documents

['Cardiovascular Disease and Risk Factors in Asia\nA Selected Review\nHirotsugu Ueshima, MD; Akira Sekikawa, MD; Katsuyuki Miura, MD;\nTanvir Chowdhury Turin, MBBS; Naoyuki Takashima, MD; Yoshikuni Kita, PhD;\nMakoto Watanabe, MD; Aya Kadota, MD; Nagako Okuda, MD; Takashi Kadowaki, MD;\nYasuyuki Nakamura, MD; Tomonori Okamura, MD\nC\nardiovascular disease (CVD) prevention in Asia is an\nimportant issue for world health, because half of the\nworld’s population lives in Asia. Asian countries and regions\nsuch as Japan, the Republic of Korea, the People’s Republic\nof China, Hong Kong, Taiwan, and the Kingdom of Thailand\nhave greater mortality and morbidity from stroke than from\ncoronary heart disease (CHD), whereas the opposite is true in\nWestern countries.\n1 The reasons why this specific situation is\nobserved in countries with rapid and early-phase westerniza-\ntion, such as Japan and South Korea, are very interesting.\nThe Seven Countries Study conducted by Keys et al\n2 in\n1957 

# For Google Colab

    from google.colab import files

    uploaded = files.upload()

In [14]:
# def read_pdf(path):

#     reader = PdfReader(path)

#     text = ""

#     for page in reader.pages:

#         page_text = page.extract_text()

#         if page_text:
#             text += page_text + "\n"

#     return text

In [15]:
# def chunk_text(text, chunk_size=500):

#     words = text.split()

#     chunks = []

#     for i in range(0, len(words), chunk_size):

#         chunk = " ".join(words[i:i + chunk_size])

#         chunks.append(chunk)

#     return chunks

In [16]:
# documents = []
# document_names = []

# chunks_1 = chunk_text(pdf_1_text)
# chunks_2 = chunk_text(pdf_2_text)

# for chunk in chunks_1:

#     documents.append(chunk)
#     document_names.append("paper1.pdf")

# for chunk in chunks_2:

#     documents.append(chunk)
#     document_names.append("paper2.pdf")

In [17]:
# documents[0]

Why chunking matters

Without chunking:

1 PDF = 1 huge embedding

With chunking:

1 PDF → many small chunks → many embeddings

This is the core idea of practical RAG systems.

In [18]:
# print("Number of chunks:", len(documents))

In [19]:
# # 3. Embed all documents
# doc_embeddings = [embed(doc) for doc in documents]

In [20]:
# # 6. Try it
# question = "What is the leading cause of cardiovascular disease?"

# answer = rag_answer(question)

# print(answer)

# RAG System Version 2.0 - Where and in which document is the information

In [21]:
from pypdf import PdfReader

def build_documents_from_pdf(path, chunk_size=50):
    reader = PdfReader(path)

    docs = []
    names = []
    pages = []
    chunks = []

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text()

        if page_text:

            words = page_text.split()

            for chunk_number, i in enumerate(range(0, len(words), chunk_size), start=1):

                chunk = " ".join(words[i:i + chunk_size])

                docs.append(chunk)
                names.append(path)
                pages.append(page_number)
                chunks.append(chunk_number)

    return docs, names, pages, chunks

In [22]:
docs1, names1, pages1, chunks1 = build_documents_from_pdf("paper1.pdf")
docs2, names2, pages2, chunks2 = build_documents_from_pdf("paper2.pdf")

documents = docs1 + docs2
document_names = names1 + names2
document_pages = pages1 + pages2
document_chunks = chunks1 + chunks2

In [23]:
def retrieve(query, k=2):

    query_embedding = embed(query)

    similarities = []

    for doc_embedding in doc_embeddings:

        score = np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) *
            np.linalg.norm(doc_embedding)
        )

        similarities.append(score)

    top_indices = np.argsort(similarities)[-k:][::-1]

    results = []

    for i in top_indices:

        results.append({
            "document_id": i,
            "document_name": document_names[i],
            "page": document_pages[i],
            "chunk": document_chunks[i],
            "document": documents[i],
            "similarity": similarities[i]
        })

    return results

In [24]:
# # Load local free model
# generator = pipeline(
#     "text-generation",
#     model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# )

def rag_answer(question):

    retrieved_docs = retrieve(question, k=2)

    context = ""

    for r in retrieved_docs:
        context += f"""
Source: {r["document_name"]}, page {r["page"]}, chunk {r["chunk"]}
Text: {r["document"]}
"""

    prompt = f"""
Answer the question using ONLY the context below.

After the answer, cite the source using:
(document name, page number, chunk number)

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.0
    )

    generated_text = response[0]["generated_text"]

    # Remove prompt from generated text
    answer = generated_text[len(prompt):].strip()

    print("=== RETRIEVED SOURCES ===\n")

    for r in retrieved_docs:
        print("DOCUMENT:", r["document_name"])
        print("PAGE:", r["page"])
        print("CHUNK:", r["chunk"])
        print("SIMILARITY:", round(r["similarity"], 3))
        print("TEXT PREVIEW:", r["document"][:500], "...")
        print()

    print("=== GENERATED ANSWER ===\n")
    print(answer)

In [25]:
question = "What is the leading cause of cardiovascular disease?"

rag_answer(question)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RETRIEVED SOURCES ===

DOCUMENT: paper1.pdf
PAGE: 1
CHUNK: 1
SIMILARITY: 0.101
TEXT PREVIEW: Cardiovascular Disease and Risk Factors in Asia A Selected Review Hirotsugu Ueshima, MD; Akira Sekikawa, MD; Katsuyuki Miura, MD; Tanvir Chowdhury Turin, MBBS; Naoyuki Takashima, MD; Yoshikuni Kita, PhD; Makoto Watanabe, MD; Aya Kadota, MD; Nagako Okuda, MD; Takashi Kadowaki, MD; Yasuyuki Nakamura, MD; Tomonori Okamura, MD C ardiovascular disease ...

DOCUMENT: paper1.pdf
PAGE: 1
CHUNK: 3
SIMILARITY: 0.1
TEXT PREVIEW: stroke than from coronary heart disease (CHD), whereas the opposite is true in Western countries. 1 The reasons why this specific situation is observed in countries with rapid and early-phase westerniza- tion, such as Japan and South Korea, are very interesting. The Seven Countries Study conducted by Keys et al ...

=== GENERATED ANSWER ===

Cardiovascular disease is the leading cause of cardiovascular disease.


# Better Version

In [ ]:
#!pip install transformers torch sentence-transformers scikit-learn

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-large" # if slow use flan-t5-base

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)




Loading weights: 100%|██████████| 558/558 [00:00<00:00, 6258.13it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [41]:
def generate_answer(prompt, max_new_tokens=250):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        outputs = llm.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=False,
    num_beams=4,
    length_penalty=1.2,
    no_repeat_ngram_size=3
)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def rag_answer(question):

    retrieved_docs = retrieve(question, k=5)

    context = ""

    for i, r in enumerate(retrieved_docs):
        context += f"""
[S{i+1}]
Document: {r["document_name"]}
Page: {r["page"]}
Chunk: {r["chunk"]}
Text: {r["document"]}
"""

    prompt = f"""
Use the context below to answer the question.

Write 3 to 5 sentences.

If the context contains partial information, explain what it says.
Do not use outside knowledge.

Always cite the source using [S1], [S2], etc.

Context:
{context}

Question:
{question}

Answer in 3 to 5 sentences:
"""

    answer = generate_answer(prompt, max_new_tokens=350)

    print("=== RETRIEVED SOURCES ===\n")

    for i, r in enumerate(retrieved_docs):
        print(f"SOURCE: [S{i+1}]")
        print("DOCUMENT:", r["document_name"])
        print("PAGE:", r["page"])
        print("CHUNK:", r["chunk"])
        print("SIMILARITY:", round(r["similarity"], 3))
        print("TEXT PREVIEW:", r["document"][:500], "...")
        print()

    print("=== GENERATED ANSWER ===\n")
    print(answer)

In [42]:
question = "What is the leading cause of cardiovascular disease?"

rag_answer(question)

=== RETRIEVED SOURCES ===

SOURCE: [S1]
DOCUMENT: paper1.pdf
PAGE: 1
CHUNK: 1
SIMILARITY: 0.101
TEXT PREVIEW: Cardiovascular Disease and Risk Factors in Asia A Selected Review Hirotsugu Ueshima, MD; Akira Sekikawa, MD; Katsuyuki Miura, MD; Tanvir Chowdhury Turin, MBBS; Naoyuki Takashima, MD; Yoshikuni Kita, PhD; Makoto Watanabe, MD; Aya Kadota, MD; Nagako Okuda, MD; Takashi Kadowaki, MD; Yasuyuki Nakamura, MD; Tomonori Okamura, MD C ardiovascular disease ...

SOURCE: [S2]
DOCUMENT: paper1.pdf
PAGE: 1
CHUNK: 3
SIMILARITY: 0.1
TEXT PREVIEW: stroke than from coronary heart disease (CHD), whereas the opposite is true in Western countries. 1 The reasons why this specific situation is observed in countries with rapid and early-phase westerniza- tion, such as Japan and South Korea, are very interesting. The Seven Countries Study conducted by Keys et al ...

SOURCE: [S3]
DOCUMENT: paper1.pdf
PAGE: 1
CHUNK: 5
SIMILARITY: 0.015
TEXT PREVIEW: an increase in dietary fat intake from 10% of total en